In [ ]:
### 통합

import os
import glob
import pandas as pd


# ===============================
# 기본 경로 (토지_매매 폴더)
# ===============================

base_folder = r'C:\startcoding\GUKTO\토지_매매'


# ===============================
# 통합폴더 생성
# ===============================

merge_folder = os.path.join(base_folder, '토지_매매_통합')
os.makedirs(merge_folder, exist_ok=True)


# ===============================
# 시도 폴더 목록
# ===============================

region_folders = [
    f for f in os.listdir(base_folder)
    if os.path.isdir(os.path.join(base_folder, f))
]


# ===============================
# 시도별 통합
# ===============================

for region in region_folders:

    folder_path = os.path.join(base_folder, region)

    print(f"\n📂 처리중: {region}")

    xlsx_files = [
        f for f in glob.glob(os.path.join(folder_path, '*.xlsx'))
        if not f.endswith('통합.xlsx')
    ]

    merged_df = pd.DataFrame()

    for file_path in xlsx_files:

        try:

            df = pd.read_excel(
                file_path,
                skiprows=13,
                header=None
            ).iloc[:, :17]

            merged_df = pd.concat(
                [merged_df, df],
                ignore_index=True
            )

            print(f"   ✅ 추가: {os.path.basename(file_path)}")

        except Exception as e:

            print(f"   ❌ 오류: {os.path.basename(file_path)} - {e}")


    # ===============================
    # 저장
    # ===============================

    output_path = os.path.join(
        merge_folder,
        f'{region}_토지_매매_통합.xlsx'
    )

    merged_df.to_excel(
        output_path,
        index=False,
        header=False,
        engine='openpyxl'
    )

    print(f"🎉 저장 완료: {output_path}")

In [1]:
###정제

import pandas as pd
import os
import glob

# =========================
# 경로
# =========================

input_folder = r'C:\startcoding\GUKTO\토지_매매\토지_매매_통합'
output_folder = r'C:\startcoding\GUKTO\토지_매매\토지_매매_정제'

os.makedirs(output_folder, exist_ok=True)

excel_files = [
    f for f in glob.glob(os.path.join(input_folder,'*.xlsx'))
    if not os.path.basename(f).startswith('~$')
]

# =========================
# 파일 처리
# =========================

for file_path in excel_files:

    print("처리중:", os.path.basename(file_path))

    df = pd.read_excel(file_path, engine='openpyxl', header=None)

    df.columns = [
        '순번','시군구','번지','지목','용도지역','도로조건',
        '계약연월','계약일','계약면적','거래금액(만원)',
        '지분구분','해제사유발생일','거래유형','중개사소재지'
    ]

    # 숫자 정리
    df['거래금액(만원)'] = (
        df['거래금액(만원)']
        .astype(str)
        .str.replace(',','')
        .astype(float)
    )

    df['계약면적'] = pd.to_numeric(df['계약면적'], errors='coerce')

    # 해제 거래 제거
    df = df[df['해제사유발생일'] == '-']

    # 중개사 제거
    df = df.drop(columns=['중개사소재지'])

    # 주소 분리
    split_columns = (
        df['시군구']
        .str.split(' ', n=4, expand=True)
        .reindex(columns=range(5))
    )

    split_columns.columns = ['주소1','주소2','주소3','주소4','주소5']

    df = pd.concat([df, split_columns], axis=1)

    # 연도 / 월
    df['연도'] = df['계약연월'].astype(str).str[:4]
    df['월'] = df['계약연월'].astype(str).str[4:]

    # =========================
    # 면적구분 (벡터연산)
    # =========================

    df['면적구분'] = '정상'

    df.loc[df['계약면적'] < 30, '면적구분'] = '광소'
    df.loc[df['계약면적'] >= 3000, '면적구분'] = '광대'

    # =========================
    # 단가
    # =========================

    df['단가'] = (df['거래금액(만원)'] / df['계약면적']).round(1)

    # 필요없는 열 제거
    df = df.drop(columns=['계약연월','해제사유발생일','계약일'])

    # 열 순서
    df = df[
        ['순번','시군구','주소1','주소2','주소3','주소4','주소5',
         '번지','지목','용도지역','도로조건',
         '연도','월',
         '면적구분','계약면적','거래금액(만원)','단가',
         '지분구분','거래유형']
    ]

    # =========================
    # 용도지역 축약
    # =========================

    zone_map = {
        '제1종전용주거지역':'1전','제2종전용주거지역':'2전',
        '제1종일반주거지역':'1주','제2종일반주거지역':'2주',
        '제3종일반주거지역':'3주','준주거지역':'준주',
        '근린상업지역':'근상','유통상업지역':'유상',
        '일반상업지역':'일상','중심상업지역':'중상',
        '전용공업지역':'전공','일반공업지역':'일공',
        '준공업지역':'준공','자연녹지지역':'자녹',
        '생산녹지지역':'생녹','보전녹지지역':'보녹',
        '계획관리지역':'계관','보전관리지역':'보관',
        '생산관리지역':'생관','개발제한구역':'개제',
        '농림지역':'농림','자연환경보전지역':'자보'
    }

    df['용도지역'] = df['용도지역'].map(zone_map).fillna('기타')

    # =========================
    # 지목 축약
    # =========================

    zone_map2 = {
        '과수원':'과','창고용지':'창','임야':'임','공장용지':'장',
        '도로':'도','잡종지':'잡','구거':'구','유지':'유',
        '학교용지':'학','묘지':'묘','철도용지':'철',
        '목장용지':'목','제방':'제','공원':'공',
        '양어장':'양','하천':'천','주차장':'차',
        '종교용지':'종','수도용지':'수','유원지':'원',
        '체육용지':'체','주유소용지':'주'
    }

    df['지목'] = df['지목'].map(zone_map2).fillna(df['지목'])

    # =========================
    # 도로조건
    # =========================

    zone_map3 = {
        '8m미만':'8미',
        '12m미만':'12미',
        '25m미만':'25미',
        '25m이상':'25이'
    }

    df['도로조건'] = df['도로조건'].map(zone_map3).fillna('-')

    # =========================
    # 저장
    # =========================

    file_name = os.path.basename(file_path).replace('통합','정제')

    save_path = os.path.join(output_folder,file_name)

    df.to_excel(save_path,index=False)
    print("완료:",file_name)

print("\n전체 정제 완료")

처리중: 경기도_토지_매매_통합.xlsx
완료: 경기도_토지_매매_정제.xlsx
처리중: 경상남도_토지_매매_통합.xlsx
완료: 경상남도_토지_매매_정제.xlsx
처리중: 경상북도_토지_매매_통합.xlsx
완료: 경상북도_토지_매매_정제.xlsx
처리중: 광주광역시_토지_매매_통합.xlsx
완료: 광주광역시_토지_매매_정제.xlsx
처리중: 대구광역시_토지_매매_통합.xlsx
완료: 대구광역시_토지_매매_정제.xlsx
처리중: 대전광역시_토지_매매_통합.xlsx
완료: 대전광역시_토지_매매_정제.xlsx
처리중: 부산광역시_토지_매매_통합.xlsx
완료: 부산광역시_토지_매매_정제.xlsx
처리중: 서울특별시_토지_매매_통합.xlsx
완료: 서울특별시_토지_매매_정제.xlsx
처리중: 세종특별자치시_토지_매매_통합.xlsx
완료: 세종특별자치시_토지_매매_정제.xlsx
처리중: 울산광역시_토지_매매_통합.xlsx
완료: 울산광역시_토지_매매_정제.xlsx
처리중: 인천광역시_토지_매매_통합.xlsx
완료: 인천광역시_토지_매매_정제.xlsx
처리중: 전라남도_토지_매매_통합.xlsx
완료: 전라남도_토지_매매_정제.xlsx
처리중: 충청남도_토지_매매_통합.xlsx
완료: 충청남도_토지_매매_정제.xlsx
처리중: 충청북도_토지_매매_통합.xlsx
완료: 충청북도_토지_매매_정제.xlsx

전체 정제 완료


In [ ]:
### 분석 엑셀파일 생성 --> 시간이 너무 오래걸림

import pandas as pd
import os
import glob
import shutil

# 경로 설정
data_folder = r'C:\startcoding\GUKTO\토지_매매\토지_매매_정제'
template_path = r'C:\startcoding\GUKTO\토지_매매\시세분석_토지_매매_템플릿.xlsx'
output_folder = r'C:\startcoding\GUKTO\토지_매매\토지_매매_분석'

os.makedirs(output_folder, exist_ok=True)
excel_files = glob.glob(os.path.join(data_folder, '*.xlsx'))

region_map = {
    '서울특별시': '서울', '부산광역시': '부산', '대구광역시': '대구',
    '인천광역시': '인천', '광주광역시': '광주', '대전광역시': '대전',
    '울산광역시': '울산', '세종특별자치시': '세종', '경기도': '경기',
    '강원특별자치도': '강원', '충청북도': '충북', '충청남도': '충남',
    '전북특별자치도': '전북', '전라남도': '전남', '경상북도': '경북',
    '경상남도': '경남', '제주특별자치도': '제주'
}

for file_path in excel_files:
    file_name = os.path.basename(file_path)
    region_full = file_name.split('_')[0]
    region_short = region_map.get(region_full, region_full)
    
    print(f"처리 중: {region_short}...")

    # 데이터 읽기 (엔진 이슈가 있을 수 있으니 기본으로 설정하되 속도가 중요하다면 calamine 설치 추천)
    df = pd.read_excel(file_path)

    new_excel_name = f'시세분석_토지_매매_{region_short}.xlsx'
    new_excel_path = os.path.join(output_folder, new_excel_name)

    # 템플릿 복사
    shutil.copy(template_path, new_excel_path)

    # ExcelWriter 설정 (에러 원인이었던 engine_kwargs 삭제)
    with pd.ExcelWriter(
        new_excel_path,
        engine='openpyxl',
        mode='a',
        if_sheet_exists='replace'
    ) as writer:
        df.to_excel(writer, sheet_name=f'{region_short}국토실', index=False)

print("모든 작업 완료!")

처리 중: 경기...


KeyboardInterrupt: 